# Naive SQIL on HumanoidMaze Large


In [ ]:
import random
import copy
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import HumanoidMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *
from causal_rl.algo.imitation.gail.core_net import ContinuousActor
from causal_rl.algo.imitation.gail.causal_gail import *
from causal_rl.algo.imitation.sqil.core_net import SQILQNetwork
from causal_rl.algo.imitation.sqil.causal_sqil import (
    SQILReplayBuffer, initialize_expert_buffer,
    rollout_sqil_episode, sac_update, soft_update,
    evaluate_sqil_policy,
)

In [ ]:
os.environ['CUDA_VISIBLE_DEVICES'] = '5'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
num_steps = 2000
hidden_dims = {'C'}

In [ ]:
# for eval: corrupted W, C hidden
eval_env = HumanoidMazePCH(env_id='humanoidmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False, success_radius=15.0)

In [ ]:
# load models
MODEL_PATH = '/home/et2842/causal/causalrl/models/nsqil_humlarge.pt'
ckpt = torch.load(MODEL_PATH, map_location=device, weights_only=False)

naive_sqil_actor = ContinuousActor(
    num_inputs=ckpt['z_dim'],
    num_outputs=ckpt['action_dim'],
    hidden_size=ckpt['hidden_size_actor'],
    std=0.0,
    action_low=float(ckpt['action_bounds_low'].min()),
    action_high=float(ckpt['action_bounds_high'].max()),
    num_blocks=ckpt['num_blocks_actor'],
    dropout=ckpt['dropout_actor'],
    layernorm=ckpt['layernorm_actor'],
).to(device)

naive_sqil_actor.load_state_dict(ckpt['state_dict'])
naive_sqil_actor.eval()

naive_sqil_Z_trim = ckpt['Z_sets']
dims = ckpt['dims']
lookback = ckpt['lookback']

naive_sqil_encode, _, _ = build_windowed_z_encoder(naive_sqil_Z_trim, dims=dims, lookback=lookback)
nsqil_policy = make_gail_policy(naive_sqil_actor, naive_sqil_encode, device=device, deterministic=True)
nsqil_policies = make_shared_policy_dict(nsqil_policy)

## Evaluation

In [ ]:
num_eval_eps = 1000
nsqil_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=nsqil_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
)

len(nsqil_returns)

In [ ]:
nsqil_episode_rewards = defaultdict(float)
for rec in nsqil_returns:
    ep = rec['episode']
    nsqil_episode_rewards[ep] += float(rec['reward'])

nsqil_rewards = [nsqil_episode_rewards[e] for e in range(num_eval_eps)]
sum(nsqil_rewards) / num_eval_eps

In [ ]:
mean_reward = np.mean(nsqil_rewards)
std_reward = np.std(nsqil_rewards)

print(f"E[Y]          = {mean_reward:.4f}")
print(f"Std[Y]        = {std_reward:.4f}")
print(f"E[Y] ± Std[Y] = {mean_reward:.4f} ± {std_reward:.4f}")

In [ ]:
# success rate: % of episodes solved in under 1000 steps
ep_lengths = defaultdict(int)
for rec in nsqil_returns:
    ep_lengths[rec['episode']] += 1

lengths = np.array([ep_lengths[e] for e in range(num_eval_eps)])
successes = lengths < num_steps
success_rate = successes.mean()
se = np.sqrt(success_rate * (1 - success_rate) / num_eval_eps)

print(f"Success rate   = {100 * success_rate:.2f}% ({successes.sum()}/{num_eval_eps} episodes)")
print(f"Std error      = {100 * se:.2f}%")

In [ ]:
# successful episode lengths
success_lengths = lengths[successes]

if len(success_lengths) > 0:
    print(f"Successful episode lengths (n={len(success_lengths)}):")
    print(f"  Mean   = {np.mean(success_lengths):.2f}")
    print(f"  Std    = {np.std(success_lengths):.2f}")
    print(f"  Median = {np.median(success_lengths):.0f}")
    print(f"  Min    = {np.min(success_lengths)}")
    print(f"  Max    = {np.max(success_lengths)}")
    print(f"  25th%  = {np.percentile(success_lengths, 25):.0f}")
    print(f"  75th%  = {np.percentile(success_lengths, 75):.0f}")
else:
    print("No episodes were solved.")